## **HELPFUL BECKMANN LAB CODE**

### **Goal**: Put this on github


In [ ]:
library(data.table)
library(ggplot2)
library(readxl)
library(biomaRt)
library(dplyr)
library(edgeR)
library(limma)
library(variancePartition)
library(matrixStats)

set.seed(2025)

In [ ]:
#Jolie's updated code
canCorAllAgainstAll_Original <- function(X, Y = X, minimum_intersect = 0) {
  library(stringr)

  # Create model matrices for each variable
  X_formulas <- lapply(str_c("~", colnames(X)), as.formula)
  X_varList <- lapply(X_formulas, function(xf) model.matrix.lm(xf, X, na.action = "na.pass")[, -1, drop = FALSE])

  Y_formulas <- lapply(str_c("~", colnames(Y)), as.formula)
  Y_varList <- lapply(Y_formulas, function(yf) model.matrix.lm(yf, Y, na.action = "na.pass")[, -1, drop = FALSE])

  # Initialize result matrix
  XY_cc <- matrix(nrow = ncol(X), ncol = ncol(Y), data = NA,
                  dimnames = list(colnames(X), colnames(Y)))

  for (ix in seq_along(X_varList)) {
    keep1 <- apply(X_varList[[ix]], 1, function(x) !any(is.na(x)))
    for (iy in seq_along(Y_varList)) {
      keep2 <- apply(Y_varList[[iy]], 1, function(x) !any(is.na(x)))
      keep <- keep1 & keep2

      x_mat <- X_varList[[ix]][keep, , drop = FALSE]
      y_mat <- Y_varList[[iy]][keep, , drop = FALSE]

      # Extra safe check: enough data, non-zero variance, and full rank
      if (sum(keep) > minimum_intersect &&
          ncol(x_mat) > 0 && ncol(y_mat) > 0 &&
          any(apply(x_mat, 2, var, na.rm = TRUE) > 0) &&
          any(apply(y_mat, 2, var, na.rm = TRUE) > 0)) {

        # Try cancor, safely
        try_result <- tryCatch({
          fit <- cancor(x_mat, y_mat)
          sqrt(mean(fit$cor^2))
        }, error = function(e) NA)

        XY_cc[ix, iy] <- try_result
      } else {
        XY_cc[ix, iy] <- NA
      }
    }
  }

  return(XY_cc)
}

In [ ]:
## OFFICIAL: FUNCTION -ALFY THE PLOTTING - Jolie's Updated Code
plot_pca_by_metadata <- function(pca, summ, info_all2, clonename, 
                                 file_prefix, form_label,
                                 type = "norm") {
  require(ggplot2)
  require(ggrepel)
  require(gridExtra)
  require(patchwork)
  require(sp)
  require(ggrastr)
  
  # True confidence levels for ±1, 2, 3 SD
  level1 <- pnorm(1) - pnorm(-1)  # ~68.27%
  level2 <- pnorm(2) - pnorm(-2)  # ~95.45%
  level3 <- pnorm(3) - pnorm(-3)  # ~99.73%

  total <- ncol(info_all2)
  count <- 1
  dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
  out_path <- paste0("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",
                     file_prefix, "_", form_label, "_", dateFreeze, "residualizedID_rin.pdf")
  
  pdf(out_path, width = 10, height = 8)
  
  for (col in colnames(info_all2)) {
    cat("on column", count, "/", total, col, "\n")
    color_data <- info_all2[[col]]
    
    plot_pair <- function(x, y, pc_x, pc_y, col_name, is_discrete) {
      base <- ggplot(data.frame(pca$x), aes_string(x = pc_x, y = pc_y,
                                                   colour = if (is_discrete) paste0("factor(info_all2[['", col_name, "']])")
                                                            else paste0("info_all2[['", col_name, "']]"),
                                                   label = "clonename")) +
        theme_bw() +
        rasterize(geom_point(size = 0.8)) +
        labs(title = paste0("PC", x, "-PC", y),
             x = paste0("PC", x, ": ", round(summ$importance[2, x] * 100, digits = 2), "%"),
             y = paste0("PC", y, ": ", round(summ$importance[2, y] * 100, digits = 2), "%")) +
        ggtitle(col_name) +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level3, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level2, linetype = "dotdash", colour = "darkgrey") +
        stat_ellipse(aes_string(x = pc_x, y = pc_y), inherit.aes = FALSE,
                     type = type, level = level1, linetype = "dotdash", colour = "darkgrey")
      
      if (is_discrete) {
        if (length(unique(color_data)) < 7) {
          base <- base +
            scale_colour_discrete(guide = "legend", name = substr(col_name, 1, 6)) +
            theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
            guides(colour = guide_legend(override.aes = list(size = 2)))
        } else {
          base <- base + scale_colour_discrete(guide = "none")
        }
      } else {
        base <- base + scale_colour_gradientn(guide = "legend",
                                               name = substr(col_name, 1, 6),
                                               colours = rainbow(2)) +
          theme(legend.key.size = unit(0.5, "cm"), legend.key.width = unit(0.2, "cm")) +
          guides(colour = guide_legend(override.aes = list(size = 2)))
      }
      return(base)
    }
    
    is_discrete <- class(color_data) != "numeric"
    
    a <- plot_pair(1, 2, "pca$x[,1]", "pca$x[,2]", col, is_discrete)
    b <- plot_pair(2, 3, "pca$x[,2]", "pca$x[,3]", col, is_discrete)
    c <- plot_pair(3, 4, "pca$x[,3]", "pca$x[,4]", col, is_discrete)
    d <- plot_pair(4, 5, "pca$x[,4]", "pca$x[,5]", col, is_discrete)

    print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count <- count + 1
  }
  
  dev.off()
  cat("PDF saved to:", out_path, "\n")
}

In [ ]:
#Jolie's updated code
level3=pnorm(3,mean=0,sd=1,lower.tail=T) - pnorm(3,lower.tail=F) #Probability that a Z-score is within ±3 standard deviations of the mean
level2=pnorm(2,mean=0,sd=1,lower.tail=T) - pnorm(2,lower.tail=F)
level1=pnorm(1,mean=0,sd=1,lower.tail=T) - pnorm(1,lower.tail=F)
type="norm"

options(width=150)
library(gridExtra)
library(patchwork)
library(sp)
pca <- pca_results
resCor=canCorAllAgainstAll_Original(blood_metadata,as.data.frame(pca$x[,1:10]),minimum_intersect=100)

##ordering in descending order
ordered_resCor=do.call(order,as.data.frame(-resCor[,1:10]))
resCor=resCor[ordered_resCor,]
blood_metadata2=blood_metadata[,rownames(resCor)] #columns of blood_metadata2 is in the same order as rownames(resCor), which is descending corr order
head(resCor,100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,2:5]))
# head(resCor[ordered_resCor,],100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,3:5]))
# head(resCor[ordered_resCor,],100)
library(ggrastr)

info_all2 <- blood_metadata2
dateFreeze <- format(Sys.Date(), "%Y-%m-%d")
summ=summary(pca)
SampleByVariable = t(v$E)
clonename<-rownames(SampleByVariable)

pdf(paste("/hpc/users/hoangd02/www/plots/lbp/pca_plots_",dateFreeze,".pdf",sep="")) #this is for blood

pdf(paste("/hpc/users/hoangd02/www/plots/lbp/pca_plots_brain_",dateFreeze,".pdf",sep="")) #this is for brain
  count=1
  total=ncol(info_all2)
  for(col in colnames(info_all2)){
    cat("on column",count,"/",total,col,"\n")
  # #=======pca-1 vs pca-2=======
    if(class(info_all2[,col])!="numeric"){
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + #geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      if(length(unique(info_all2[,col]))<7){
        a = a + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        a = a + scale_colour_discrete(guide ="none")
      }
      build <- ggplot_build(a)$data
      points <- build[[1]]
      ell <- build[[3]]

      # Find which points are inside the ellipse, and add this to the data
      dat <- data.frame(points[1:2], 
                        in.ell = as.logical(point.in.polygon(points$x, points$y, ell$x, ell$y)))
      outliers_3SD_PC1_PC2=points$label[which(dat$in.ell==F)]
      # show(a)
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + 
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        b = b + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        b = b + scale_colour_discrete(guide ="none")
      }
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + 
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        c = c + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        c = c + scale_colour_discrete(guide ="none")
      }
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) + #geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        d = d + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        d = d + scale_colour_discrete(guide ="none")
      }
    }else{
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      a = a +
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      b = b + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      c = c + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      d = d + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
    }
    # show(a)
    # show(b)
    # show(c)
    # show(d)
   #multiplot_same_legend(a,b,c,d,cols=2)
   (a + b) / (c + d) + plot_layout(guides = "collect")
   print((a + b) / (c + d) + plot_layout(guides = "collect"))
    count=count+1
  }
dev.off()

save.image(file = "/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_with_corr_and_pc.RData")

load("/sc/arion/projects/mscic1/results/jolie/LBP/blood-brain/qc_with_corr_and_pc.RData")

https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_2025-05-08.pdf
https://hoangd02.u.hpc.mssm.edu/plots/lbp/pca_plots_2025-05-08.png

In [ ]:
## NB original code
SampleByVariable=t(cov(v$E)) ## is transpose redundant since covariance matrix is symmetric?
test<-cov(v$E) 
#tells you how similar samples are to each other based on their expression profiles across all genes

# SampleByVariable=t(vobj$E)
clonename<-rownames(SampleByVariable)
pca <- prcomp(SampleByVariable, scale=T)
summ=summary(pca)

#not sure why run PCA on a covariance matrix — 
#PCA assumes a samples × variables matrix (i.e., rows = samples, columns = genes/features)
Importance of components:
                           PC1     PC2     PC3     PC4     PC5    PC6     PC7
Standard deviation     15.4448 1.48143 1.23377 0.45493 0.37906 0.3111 0.25184
Proportion of Variance  0.9817 0.00903 0.00626 0.00085 0.00059 0.0004 0.00026
Cumulative Proportion   0.9817 0.99068 0.99695 0.99780 0.99839 0.9988 0.99905
                           PC8     PC9    PC10    PC11    PC12    PC13    PC14
Standard deviation     0.19703 0.18612 0.14672 0.13759 0.12104 0.11587 0.09696
Proportion of Variance 0.00016 0.00014 0.00009 0.00008 0.00006 0.00006 0.00004
Cumulative Proportion  0.99921 0.99935 0.99944 0.99952 0.99958 0.99963 0.99967
                          PC15    PC16    PC17    PC18    PC19    PC20    PC21
Standard deviation     0.08794 0.07475 0.07022 0.06443 0.05647 0.05459 0.05181
Proportion of Variance 0.00003 0.00002 0.00002 0.00002 0.00001 0.00001 0.00001
Cumulative Proportion  0.99971 0.99973 0.99975 0.99977 0.99978 0.99979 0.99980
                          PC22    PC23    PC24    PC25    PC26    PC27    PC28
Standard deviation     0.04904 0.04498 0.04327 0.04137 0.03923 0.03722 0.03676
Proportion of Variance 0.00001 0.00001 0.00001 0.00001 0.00001 0.00001 0.00001
Cumulative Proportion  0.99981 0.99982 0.99983 0.99984 0.99984 0.99985 0.99985

In [ ]:
## NB original code
canCorAllAgainstAll_Original <- function(X, Y = X,minimum_intersect=0) {
    # Compute canonical correlation of all columns of X against all columns of Y,
    # similar to variancePartition::canCorPairs.
    library(stringr)
    X_formulas <- lapply(str_c("~", colnames(X)), as.formula)
    X_varList <- lapply(X_formulas, function(xf) model.matrix.lm(xf, X, na.action = "na.pass")[,-1, drop = FALSE])
    Y_formulas <- lapply(str_c("~", colnames(Y)), as.formula)
    Y_varList <- lapply(Y_formulas, function(yf) model.matrix.lm(yf, Y, na.action = "na.pass")[,-1, drop = FALSE])
    XY_cc <- matrix(nrow = ncol(X), ncol = ncol(Y), data = 0,
                    dimnames = list(colnames(X), colnames(Y)))
    for (ix in seq_along(X_varList)) {
        keep1 = apply(X_varList[[ix]], 1, function(x) !any(is.na(x)))
        for (iy in seq_along(Y_varList)) {
            keep2 = apply(Y_varList[[iy]], 1, function(x) !any(is.na(x)))
            keep = keep1 & keep2
            if(sum(keep)>minimum_intersect){
                fit <- cancor(X_varList[[ix]][keep, , drop = FALSE], Y_varList[[iy]][keep, , drop = FALSE])
                # Using root-mean-square to summarize, as discussed with Gabriel Hoffman
                XY_cc[ix,iy] <- sqrt(mean(fit$cor^2))
            }else{
                XY_cc[ix,iy] <- NA
            }
        }
    }
    return(XY_cc)
}

In [ ]:
## this is the original NB code
SampleByVariable=t(cov(v$E)) 

# SampleByVariable=t(vobj$E)
clonename<-rownames(SampleByVariable)
pca <- prcomp(SampleByVariable, scale=T)
summ=summary(pca)

level3=pnorm(3,mean=0,sd=1,lower.tail=T) - pnorm(3,lower.tail=F)
level2=pnorm(2,mean=0,sd=1,lower.tail=T) - pnorm(2,lower.tail=F)
level1=pnorm(1,mean=0,sd=1,lower.tail=T) - pnorm(1,lower.tail=F)
type="norm"

options(width=150)
library(gridExtra)
library(sp)

resCor=canCorAllAgainstAll_Original(info_all2,as.data.frame(pca$x[,1:5]),minimum_intersect=100)

ordered_resCor=do.call(order,as.data.frame(-resCor[,1:5]))
resCor=resCor[ordered_resCor,]
info_all2=info_all2[,rownames(resCor)]
head(resCor,100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,2:5]))
# head(resCor[ordered_resCor,],100)
# ordered_resCor=do.call(order,as.data.frame(-resCor[,3:5]))
# head(resCor[ordered_resCor,],100)
library(ggrastr)

pdf(paste("~/www/plots/pca_plot_vobj_LBP_all_for_neuropath_",dateFreeze,".pdf",sep=""))
  count=1
  total=ncol(info_all2)
  for(col in colnames(info_all2)){
    cat("on column",count,"/",total,col,"\n")
  # #=======pca-1 vs pca-2=======
    if(class(info_all2[,col])!="numeric"){
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      if(length(unique(info_all2[,col]))<7){
        a = a + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        a = a + scale_colour_discrete(guide =FALSE)
      }
      build <- ggplot_build(a)$data
      points <- build[[1]]
      ell <- build[[3]]

      # Find which points are inside the ellipse, and add this to the data
      dat <- data.frame(points[1:2], 
                        in.ell = as.logical(point.in.polygon(points$x, points$y, ell$x, ell$y)))
      outliers_3SD_PC1_PC2=points$label[which(dat$in.ell==F)]
      # show(a)
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        b = b + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        b = b + scale_colour_discrete(guide =FALSE)
      }
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        c = c + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        c = c + scale_colour_discrete(guide =FALSE)
      }
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=factor(info_all2[,col]), label=clonename)) + theme_bw() +
        rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey") 
      if(length(unique(info_all2[,col]))<7){
        d = d + scale_colour_discrete(guide ="legend",name=substring(col, 1, 6)) +
        theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
        theme(legend.key.width=unit(0.2,"cm"))
      }else{
        d = d + scale_colour_discrete(guide =FALSE)
      }
    }else{
      a <- ggplot(data.frame(pca$x), aes(x= pca$x[,1], y= pca$x[,2], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC1-PC2") + xlab(paste("PC1: ",round(summ$importance[2,1]*100,digits=2),"%",sep="")) +
        ylab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) + ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,1],y=pca$x[,2]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      a = a +
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-2 vs pca-3=======
      b <- ggplot(data.frame(pca$x), aes(x= pca$x[,2], y= pca$x[,3], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC2-PC3")+ xlab(paste("PC2: ",round(summ$importance[2,2]*100,digits=2),"%",sep="")) +
        ylab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,2],y=pca$x[,3]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      b = b + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-3 vs pca-4=======
      c <- ggplot(data.frame(pca$x), aes(x= pca$x[,3], y= pca$x[,4], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC3-PC4")+ xlab(paste("PC3: ",round(summ$importance[2,3]*100,digits=2),"%",sep="")) +
        ylab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,3],y=pca$x[,4]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      c = c + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
      # #=======pca-4 vs pca-5=======
      d <- ggplot(data.frame(pca$x), aes(x= pca$x[,4], y= pca$x[,5], colour=(info_all2[,col]), label=clonename)) + theme_bw() +
        scale_colour_gradientn(guide ="legend",name=substring(col, 1, 6),colours=rainbow(2)) +rasterize(geom_point(size=0.8)) +#geom_text(aes(label=clonename),hjust=0, vjust=0,size=1.2)+
        labs(title="PC4-PC5")+ xlab(paste("PC4: ",round(summ$importance[2,4]*100,digits=2),"%",sep="")) +
        ylab(paste("PC5: ",round(summ$importance[2,5]*100,digits=2),"%",sep=""))+ ggtitle(paste(col)) + 
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level3,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level2,linetype = "dotdash",colour="darkgrey") +
        stat_ellipse(aes(x = pca$x[,4],y=pca$x[,5]),inherit.aes=F,type=type,level=level1,linetype = "dotdash",colour="darkgrey")  
      d = d + 
      theme(legend.key.size=unit(0.5,"cm")) + guides(colour = guide_legend(override.aes = list(size=2))) +
      theme(legend.key.width=unit(0.2,"cm"))
    }
    # show(a)
    # show(b)
    # show(c)
    # show(d)
    multiplot_same_legend(a,b,c,d,cols=2)
    count=count+1
  }
dev.off()